In [1]:
from pathlib import Path
import sys
import pyrpl
from pyrpl import Pyrpl

repo = Path(
    r"C:\Users\imaq\Projects\wwlyn-pyrpl-test\pyrpl_change"
)
bitstream = repo / "pyrpl" / "fpga" / "red_pitaya.bin"

print("Working directory:", Path.cwd())
print("Python:", sys.executable)
print("PyRPL:", pyrpl.__file__)
print("Bitstream:", bitstream)
assert bitstream.is_file()

p = Pyrpl(
    config="rp-f0fcd9",
    hostname="rp-f0fcd9.local",
    filename=str(bitstream),
    loglevel="info",
    gui=False,
    reloadfpga=True,
    reloadserver=True,
)

rp = p.rp

Working directory: c:\Users\imaq\Projects\wwlyn-pyrpl-test\pyrpl_change_python39_clean
Python: c:\Users\imaq\Projects\wwlyn-pyrpl-test\pyrpl_change_python39_clean\.venv\Scripts\python.exe
PyRPL: c:\Users\imaq\Projects\wwlyn-pyrpl-test\pyrpl_change_python39_clean\pyrpl\__init__.py
Bitstream: C:\Users\imaq\Projects\wwlyn-pyrpl-test\pyrpl_change_python39_clean\pyrpl\fpga\red_pitaya.bin


INFO:pyrpl.redpitaya:Successfully connected to Redpitaya with hostname rp-f0fcd9.local.


In [2]:
%gui qt
p.show_gui()

In [2]:
pid = p.rp.pid0
print(pid.inputfilter)

[0, 0, 0]


## Test 1: DC gain and offset calibration

PID0 is used as a programmable DC source on OUT1 by setting `p = 0`, `i = 0`, and changing `ival`. OUT1 is looped back to IN1 and is also measured with the Rigol. PID1 provides a second internal reading of IN1 but does not drive an output.

The sweep tests both polarities at RP commands `[-0.5, -0.25, 0, 0.25, 0.5]`. Comparing `src.current_output_signal` with the corresponding Rigol voltage separates multiplicative gain error from zero offset. The measured Rigol values `[-0.572, -0.282, 0.00567, 0.284, 0.572] V` give the least-squares output-path fit:

`V_rigol = 1.14142 * V_rp_output + 0.001534`

This is approximately +14.14% gain error and +1.53 mV zero offset for this board, output channel, probe, cabling, termination, and jumper configuration. The fit has about 2.69 mV RMS residual. Because the loopback also measured IN1, an input/readback fit can be calculated separately: `V_rigol = 1.14380 * V_rp_in1 + 0.004668`. Keep the output and input fits separate—the DAC/output path and ADC/input path are not the same calibration.

In [8]:
import time
import numpy as np

# PID0: DC output source
src = rp.pid0
src.input  = "in1"
src.output_direct = "out1"
src.p = 0
src.i = 0
src.setpoint = 0
src.ival = 0
src.min_voltage = -0.99
src.max_voltage = 0.99
src.paused = False

#PID1: reads IN1 through PID path, outputs nowhere
mon = rp.pid1
mon.input = "in1"
mon.output_direct = "off"
mon.p = 1
mon.i = 0
mon.setpoint = 0
mon.ival = 0
mon.min_voltage = -0.99
mon.max_voltage = 0.99
mon.inputfilter = [0,0,0,0]
mon.paused = False

In [16]:
src.ival = 0.5

In [9]:
for v in [-0.5, -0.25, 0.0, 0.25, 0.5]:
    src.ival = v
    time.sleep(0.5)

    scope_vals =[]
    pidmon_vals = []
    for _ in range(200):
        scope_vals.append(rp.scope.voltage_in1)
        pidmon_vals.append(mon.current_output_signal)
        time.sleep(0.005)
    print(
        "src_ival = ", v,
        "src_out = ", round(src.current_output_signal,4),
        "scope_in1_mean = ", round(float(np.mean(scope_vals)),4),
        "pid1_monitor_mean = ",  round(float(np.mean(pidmon_vals)),4),
        "pid1_std = ", round(float(np.std(pidmon_vals)),5),)

src_ival =  -0.5 src_out =  -0.5001 scope_in1_mean =  -0.5021 pid1_monitor_mean =  -0.5022 pid1_std =  0.00072
src_ival =  -0.25 src_out =  -0.25 scope_in1_mean =  -0.2516 pid1_monitor_mean =  -0.2518 pid1_std =  0.00103
src_ival =  0.0 src_out =  0.0 scope_in1_mean =  -0.003 pid1_monitor_mean =  -0.003 pid1_std =  0.00073
src_ival =  0.25 src_out =  0.25 scope_in1_mean =  0.2466 pid1_monitor_mean =  0.2467 pid1_std =  0.00072
src_ival =  0.5 src_out =  0.5001 scope_in1_mean =  0.4964 pid1_monitor_mean =  0.4964 pid1_std =  0.00072


## Test 2: dynamic triangle-wave verification

PID0 is first disconnected from OUT1 so its previous DC `ival` cannot add to the waveform. ASG0 then generates a zero-centered 1 kHz triangle on OUT1 while IN1 samples the loopback. The Rigol measurement checks that the DC calibration from Test 1 also describes a changing signal.

For a zero-centered triangle, `V_rms = V_peak / sqrt(3)`. Here the Rigol Max readings provide a useful dynamic sanity check: the 0.3 and 0.4 RP-unit points are within a few millivolts of the DC fit, while the 0.8 point is close enough to the DAC rail to show compression. The software-polled RP means must not be treated as DC offsets because 0.5 ms polling aliases a 1 kHz waveform. Since the Rigol Min/Vpp values were not captured and its RMS and Max statistics disagree somewhat, Test 2 validates dynamics but is not included in the affine calibration fit.

In [17]:
rp.pid0.output_direct = "off"

In [27]:
asg = rp.asg0

asg.output_direct = "out1"
asg.waveform = "ramp"
asg.frequency = 1000 # 1 KHz
asg.amplitude = 0.4
asg.offset = 0.0
asg.trigger_source = "immediately"

In [28]:
vals = []
for _ in range(5000):
    vals.append(rp.scope.voltage_in1)
    time.sleep(0.0005)
vals = np.array(vals)

print("RP mean = ", np.mean(vals))
print("RP min = ", np.min(vals))
print("RP max = ", np.max(vals))
print("RP Vpp = ", np.max(vals)-np.min(vals))

RP mean =  -0.0475296630859375
RP min =  -0.4014892578125
RP max =  0.3994140625
RP Vpp =  0.8009033203125


PID Controller

In [2]:
# Test 1 output calibration: RP OUT1 command/readback -> Rigol volts.
RP_OUT_TO_RIGOL_GAIN = 1.14142
RP_OUT_TO_RIGOL_OFFSET = 0.001534

def rp_to_rigol(v_rp):
    return RP_OUT_TO_RIGOL_GAIN * v_rp + RP_OUT_TO_RIGOL_OFFSET

def rigol_to_rp(v_rigol):
    return (v_rigol - RP_OUT_TO_RIGOL_OFFSET) / RP_OUT_TO_RIGOL_GAIN

# IN1 ADC/readback calibration: use for IN1 displays and digital PID setpoints.
RP_IN1_TO_RIGOL_GAIN = 1.14380
RP_IN1_TO_RIGOL_OFFSET = 0.004668

def rp_in1_to_rigol(v_rp_in1):
    return RP_IN1_TO_RIGOL_GAIN * v_rp_in1 + RP_IN1_TO_RIGOL_OFFSET

def rigol_to_rp_in1(v_rigol):
    return (v_rigol - RP_IN1_TO_RIGOL_OFFSET) / RP_IN1_TO_RIGOL_GAIN

In [15]:
pid = p.rp.pid0

pid.input = "in1"
pid.output_direct = "out1"
#pid.inputfilter = [0,0,0,0]
pid.min_voltage, pid.max_voltage = -1, 1
pid.pause_gains = "pi"
pid.paused = False

#K=1
pid.p = -0.3
pid.i = -180000
pid.ival = 0

desired_scope_voltage = 0.5
pid.setpoint = rigol_to_rp_in1(desired_scope_voltage)

print("raw RP setpoint = ", pid.setpoint)
print("scope-corrected setpoint = ", rp_in1_to_rigol(pid.setpoint))
print("execute!")

raw RP setpoint =  0.43310546875
scope-corrected setpoint =  0.5000540351562499
execute!


In [ ]:
IVAL_MIN = -1
IVAL_MAX = 1
if pid.ival< IVAL_MIN:
    pid.ival = IVAL_MIN
elif pid.ival > IVAL_MAX:
    pid.ival = IVAL_MAX

In [ ]:
import time
for _ in range(900):
    raw_in = rp.scope.voltage_in1
    corrected_in = rp_in1_to_rigol(raw_in)
    corrected_set = rp_in1_to_rigol(pid.setpoint)

    print(
        "raw in = ", round(raw_in,4),
        "scope in = ", round(corrected_in,4),
        "scope_set = ", round(corrected_set,4),
        "out = ", round(pid.current_output_signal,4),
        "ival = ", round(pid.ival,4)
    )
    time.sleep(0.1)

## Analog setpoint PID mode

Connect the measured process signal to IN1 and the external analog reference to IN2. PID0 supplies the internal reference path without driving an output. PID1 computes `error = filtered(IN1) - filtered(IN2)` in differential mode and continuously drives OUT1.

Do not insert any of the Python calibration helpers above into this analog setpoint feedback path. The subtraction runs continuously in the FPGA, so Python is not involved sample by sample. A calibration shared by IN1 and IN2 preserves their equality point; if their gains or offsets differ, calibrate the two ADC channels separately and compensate in hardware or the analog front end. The output helpers are still useful for displaying physical output voltage or translating physical OUT1 limits into RP output units.

This example does not configure lock/hold or any digital control pin. Check the feedback polarity, gains, voltage limits, wiring, and actuator safety before executing it. The digital `pid.setpoint` value is ignored in differential mode.

In [3]:
# Reference path: IN2 -> PID0 filtered input; no direct output.
analog_reference = rp.pid0
analog_pid = rp.pid1
analog_reference.input = "in2"
analog_reference.inputfilter = []  # Disable all stages; match both differential paths.
analog_reference.output_direct = "off"
analog_reference.p = 0
analog_reference.i = 0
analog_reference.ival = 0
analog_reference.paused = False

# Controller path: error = IN1 - IN2, 
analog_pid.input = "in1"
analog_pid.inputfilter = []  # Must match the reference path.
analog_pid.differential_mode_enabled = True
analog_pid.output_direct = "out1"
analog_pid.min_voltage = -1
analog_pid.max_voltage = 1
analog_pid.p = -0  # Verify the sign for the physical plant.
analog_pid.i = -400000
analog_pid.ival = 0

rp.hk.expansion_P1_output = False  # Configure DIO1_P as input
analog_pid.pause_gains = "pi"      # Freeze both P and I
analog_pid.paused = False     
#analog_pid.pause_gains = "off"  # Lock/hold disabled.

print("Continuous analog-setpoint PID enabled; lock/hold is disabled.")

Continuous analog-setpoint PID enabled; lock/hold is disabled.


Turn Off

In [14]:
for name in ["asg0", "asg1", "pid0", "pid1","pid2", "iq0", "iq1", "iq2", "iir"]:
 try:
     m = getattr(rp, name)
     m.output_direct = "off"
     print(name, "-> off")
 except Exception as e:
    print(name, "skip: ", e)

asg0 -> off
asg1 -> off
pid0 -> off
pid1 -> off
pid2 -> off
iq0 -> off
iq1 -> off
iq2 -> off
iir skip:  'RedPitaya' object has no attribute 'iir'


In [ ]:
# Drive OUT1 at +1.000 V DC using the measured OUT1 calibration above.
target_out1_voltage = 1.0
rp_dc_offset = rigol_to_rp(target_out1_voltage)

if not -1.0 <= rp_dc_offset <= 1.0:
    raise ValueError(
        f"Calibrated OUT1 command {rp_dc_offset:.6f} V is outside the ASG range."
    )

# OUT1 sums routed FPGA modules, so disconnect every other source first.
for name in ["asg1", "pid0", "pid1", "pid2", "iq0", "iq1", "iq2"]:
    getattr(rp, name).output_direct = "off"

dc_out = rp.asg0
dc_out.setup(
    waveform="dc",
    amplitude=0.0,
    offset=rp_dc_offset,
    trigger_source="immediately",
    output_direct="out1",
)

print(
    f"OUT1 set to {target_out1_voltage:.3f} V DC "
    f"(calibrated RP command: {rp_dc_offset:.6f} V)."
)